### This book shows how we fill library, and here we will collect results 

What should our tensors include?
Features which describe size of catenary curves, transmissions and phase shift.

Подумаем от каких параметров меняется форма ячейки.
1. Мы можем изменять высоту ячейки H, но это не типичный параметр для вариации
2. Мы можем менять изгиб 'a', что влияет на скорость смещения центра. Мы можем изменять зависимость расстояния между верхней и нижней кривой. Сейчас это константа для одной формы. Если это не будет константой, то нужно будет менять способ расчета сечений.
Итого, получаем высота - не изменяемый параметр для всей библиотеки. y_center зависит а. Итого получаем 2 изменяемых параметра при текущей конфигурации.     

In [9]:
import meep as mp
import conf
import numpy as np
import math

In [10]:
pml_layers = [mp.PML(thickness=conf.in_pml, direction=mp.X)]
cell = mp.Vector3(conf.cell_X, conf.cell_Y, 0)
symmetries = [mp.Mirror(mp.Y)]
geometry = [mp.Block(
    material=conf.in_material,
    size=mp.Vector3(conf.in_sub, mp.inf, conf.cell_Z),
    center=conf.sub_center,
)]
sources = [mp.Source(
    mp.ContinuousSource(conf.inc_fcen, fwidth=conf.inc_df),
    component=mp.Ez,
    center=conf.source_center,
    size=mp.Vector3(x=0, y=conf.cell_Y, z=conf.cell_Z),
)]

sim = mp.Simulation(
    cell_size=cell,
    boundary_layers=pml_layers,
    geometry=geometry,
    sources=sources,
    resolution=conf.resolution,
    symmetries = symmetries,
)
flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(center=conf.flux_center, size=mp.Vector3(y=conf.cell_Y, z=conf.cell_Z))
)
sim.run(until=conf.steps) # run 1st part of simulation
input_flux = mp.get_fluxes(flux_obj) # saving flux from sub in input_flux
sim.reset_meep() # purge all adjustments of simulation

-----------
Initializing structure...
Halving computational cell along direction y
time for choose_chunkdivision = 0.00139904 s
Working in 2D dimensions.
Computational cell is 8 x 5 x 0 with resolution 20
     block, center = (-1,0,2)
          size (1,1e+20,4)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
time for set_epsilon = 0.00645208 s
-----------


FloatProgress(value=0.0, description='0% done ', max=50.0)

run 0 finished at t = 50.0 (2000 timesteps)


In [11]:
step = 500
mode_tran = np.empty(step)
mode_phase = np.empty(step)
size_block = [None] * step  # Список для объектов mp.Vector3
min_width, max_width = 0.3, 2
min_a, max_a = 0.5, 3

# Параметры цепной линии
a = np.linspace(min_a, max_a, step)  # параметр формы цепной линии
width = np.linspace(min_width, max_width, step) # параметр ширины цепной линии
x_max = 1  # диапазон x
x_min = -x_max

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

In [12]:
width[4]

0.313627254509018

In [13]:
number_sections = 1

mp.Verbosity(0)
for j in range(step):
    x_sections = np.linspace(width[j]/2, x_max, number_sections, endpoint=False) # В этих точках будем вычислять значения для y1 and y2  
    for i in range(number_sections): #фактически номер секции сейчас не используется, и это просто заготовка. Внутри код где-то это не учитывает поэтому работаем с одной секцией
        if x_sections[i] <= a[j]:
            x_centers = 0
            width_wide = x_sections[i]*2     
        else:
            find_width = (a[j] * math.acosh(math.cosh(x_sections[i] / a[j]) - width[j] / a[j])) - x_sections[i]
            width_wide = find_width 
            x_centers = x_sections[i] + width/4
        
        geometry[1].size = mp.Vector3(conf.at_hei, width_wide, 0)
        geometry[1].center = mp.Vector3(y=x_centers)
        geometry[2].size = mp.Vector3(conf.at_hei, width_wide, 0)
        geometry[2].center =mp.Vector3(y=-x_centers)
        
        sim = mp.Simulation(
            cell_size=cell,
            boundary_layers=pml_layers,
            geometry=geometry,
            sources=sources,
            resolution=conf.resolution,
        )
        flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(
                center=mp.Vector3(x=conf.flux_center[0]), 
                size=mp.Vector3(y=conf.cell_Y)
            )
        )
        sim.run(until=conf.steps)
        
        res = sim.get_eigenmode_coefficients(
            flux_obj, [1], eig_parity=mp.ODD_Z + mp.EVEN_Y
        )
        coeffs = res.alpha
        
        mode_tran[j] = abs(coeffs[0, 0, 0]) ** 2 / input_flux[0]
        mode_phase[j] = np.angle(coeffs[0, 0, 0])
        if mode_phase[j] > 0:
            mode_phase[j] -= 2 * np.pi
        
        
        '''
        %matplotlib inline
        plt.figure(dpi=100)
        sim.plot2D(fields=mp.Ez)
        plt.show()
        sim.reset_meep() # purge all adjustments of simulation
        '''

-----------
Initializing structure...
time for choose_chunkdivision = 4.41074e-05 s
Working in 2D dimensions.
Computational cell is 8 x 5 x 0 with resolution 20
     block, center = (-1,0,2)
          size (1,1e+20,4)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.3,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.3,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
time for set_epsilon = 0.0159121 s
-----------


FloatProgress(value=0.0, description='0% done ', max=50.0)

run 0 finished at t = 50.0 (2000 timesteps)
MPB solved for frequency_1(0.588348,0,0) = 0.579098 after 17 iters
MPB solved for frequency_1(0.506708,0,0) = 0.499705 after 11 iters
MPB solved for frequency_1(0.50701,0,0) = 0.5 after 7 iters
MPB solved for frequency_1(0.50701,0,0) = 0.5 after 1 iters
Dominant planewave for band 1: (0.507010,-0.000000,0.000000)
-----------
Initializing structure...
time for choose_chunkdivision = 3.29018e-05 s
Working in 2D dimensions.
Computational cell is 8 x 5 x 0 with resolution 20
     block, center = (-1,0,2)
          size (1,1e+20,4)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.303407,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.303407,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
time for se

FloatProgress(value=0.0, description='0% done ', max=50.0)

run 0 finished at t = 50.0 (2000 timesteps)
MPB solved for frequency_1(0.588348,0,0) = 0.57895 after 17 iters
MPB solved for frequency_1(0.50681,0,0) = 0.4997 after 11 iters
MPB solved for frequency_1(0.507118,0,0) = 0.5 after 7 iters
MPB solved for frequency_1(0.507118,0,0) = 0.5 after 1 iters
Dominant planewave for band 1: (0.507118,-0.000000,0.000000)
-----------
Initializing structure...
time for choose_chunkdivision = 4.19617e-05 s
Working in 2D dimensions.
Computational cell is 8 x 5 x 0 with resolution 20
     block, center = (-1,0,2)
          size (1,1e+20,4)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.306814,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,0.306814,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
time for set_

FloatProgress(value=0.0, description='0% done ', max=50.0)

... (991 outputs left out)

FloatProgress(value=0.0, description='0% done ', max=50.0)

run 0 finished at t = 50.0 (2000 timesteps)
MPB solved for frequency_1(0.588348,0,0) = 0.528345 after 15 iters
MPB solved for frequency_1(0.555383,0,0) = 0.499982 after 8 iters
MPB solved for frequency_1(0.555404,0,0) = 0.5 after 4 iters
MPB solved for frequency_1(0.555404,0,0) = 0.5 after 1 iters
Dominant planewave for band 1: (0.555404,-0.000000,0.000000)
-----------
Initializing structure...
time for choose_chunkdivision = 5.6982e-05 s
Working in 2D dimensions.
Computational cell is 8 x 5 x 0 with resolution 20
     block, center = (-1,0,2)
          size (1,1e+20,4)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,2,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
     block, center = (0,0,0)
          size (2,2,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (2,2,2)
time for set_epsilon = 0.

FloatProgress(value=0.0, description='0% done ', max=50.0)

run 0 finished at t = 50.0 (2000 timesteps)
MPB solved for frequency_1(0.588348,0,0) = 0.528305 after 15 iters
MPB solved for frequency_1(0.555428,0,0) = 0.499982 after 8 iters
MPB solved for frequency_1(0.555449,0,0) = 0.5 after 4 iters
MPB solved for frequency_1(0.555449,0,0) = 0.5 after 1 iters
Dominant planewave for band 1: (0.555449,-0.000000,0.000000)


In [14]:
mode_phase, mode_tran

(array([-1.86293353, -1.85916631, -1.85526688, -1.85122741, -1.84703945,
        -1.8426939 , -1.83818088, -1.83348972, -1.82860891, -1.82352596,
        -1.81822741, -1.81269874, -1.80692434, -1.80088745, -1.79457023,
        -1.78879821, -1.78612271, -1.78338235, -1.78057434, -1.77769573,
        -1.77474336, -1.77171387, -1.76860369, -1.765409  , -1.7621257 ,
        -1.75874942, -1.75527547, -1.75169884, -1.74801411, -1.74421547,
        -1.74029668, -1.73625099, -1.73207112, -1.7277492 , -1.72327673,
        -1.71864447, -1.71384244, -1.70885978, -1.70368471, -1.69830442,
        -1.692705  , -1.68687129, -1.68078687, -1.67443385, -1.66779292,
        -1.66399122, -1.66112048, -1.65818227, -1.65517363, -1.6520914 ,
        -1.64893221, -1.64569247, -1.6423683 , -1.63895558, -1.63544985,
        -1.63184633, -1.62813988, -1.62432492, -1.62039547, -1.616345  ,
        -1.61216647, -1.6078522 , -1.60339385, -1.59878229, -1.59400758,
        -1.58905878, -1.58392391, -1.57858977, -1.5

In [22]:
width[:10]

array([0.3       , 0.31717172, 0.33434343, 0.35151515, 0.36868687,
       0.38585859, 0.4030303 , 0.42020202, 0.43737374, 0.45454545,
       0.47171717, 0.48888889, 0.50606061, 0.52323232, 0.54040404,
       0.55757576, 0.57474747, 0.59191919, 0.60909091, 0.62626263,
       0.64343434, 0.66060606, 0.67777778, 0.69494949, 0.71212121,
       0.72929293, 0.74646465, 0.76363636, 0.78080808, 0.7979798 ,
       0.81515152, 0.83232323, 0.84949495, 0.86666667, 0.88383838,
       0.9010101 , 0.91818182, 0.93535354, 0.95252525, 0.96969697,
       0.98686869, 1.0040404 , 1.02121212, 1.03838384, 1.05555556,
       1.07272727, 1.08989899, 1.10707071, 1.12424242, 1.14141414,
       1.15858586, 1.17575758, 1.19292929, 1.21010101, 1.22727273,
       1.24444444, 1.26161616, 1.27878788, 1.2959596 , 1.31313131,
       1.33030303, 1.34747475, 1.36464646, 1.38181818, 1.3989899 ,
       1.41616162, 1.43333333, 1.45050505, 1.46767677, 1.48484848,
       1.5020202 , 1.51919192, 1.53636364, 1.55353535, 1.57070

In [23]:
a[:10]

array([0.5       , 0.52525253, 0.55050505, 0.57575758, 0.6010101 ,
       0.62626263, 0.65151515, 0.67676768, 0.7020202 , 0.72727273,
       0.75252525, 0.77777778, 0.8030303 , 0.82828283, 0.85353535,
       0.87878788, 0.9040404 , 0.92929293, 0.95454545, 0.97979798,
       1.00505051, 1.03030303, 1.05555556, 1.08080808, 1.10606061,
       1.13131313, 1.15656566, 1.18181818, 1.20707071, 1.23232323,
       1.25757576, 1.28282828, 1.30808081, 1.33333333, 1.35858586,
       1.38383838, 1.40909091, 1.43434343, 1.45959596, 1.48484848,
       1.51010101, 1.53535354, 1.56060606, 1.58585859, 1.61111111,
       1.63636364, 1.66161616, 1.68686869, 1.71212121, 1.73737374,
       1.76262626, 1.78787879, 1.81313131, 1.83838384, 1.86363636,
       1.88888889, 1.91414141, 1.93939394, 1.96464646, 1.98989899,
       2.01515152, 2.04040404, 2.06565657, 2.09090909, 2.11616162,
       2.14141414, 2.16666667, 2.19191919, 2.21717172, 2.24242424,
       2.26767677, 2.29292929, 2.31818182, 2.34343434, 2.36868

### Fine, now we want to save result as database

In [24]:
print(mode_tran.dtype)
print(mode_tran.ndim)

float64
1


In [25]:
# Объединяем в один двумерный массив
data = np.column_stack((mode_phase, mode_tran, width, a))

# Сохраняем в файл в формате CSV
np.savetxt('vectors1.csv', data, delimiter=',', header='mode_phase,mode_tran,width,a', comments='', fmt='%.6f')

# Альтернатива: сохранить в бинарном формате .npy
np.save('vectors1.npy', data)

print("Данные успешно сохранены!")

Данные успешно сохранены!


In [26]:
data.ndim

2

In [27]:
data.size

400

In [28]:
data.dtype

dtype('float64')

Finally. Проверка завершена. Данные для обучения расчитываются и копятся корректно.
### TODO: Реализовать мультипоточность через параллелизм независимых процессов.

Теперь надо прикинуть, что делаем дальше. Мы имеем расчитанный набор данных. Но их еще нужно подготовить для обучения.
Нужно разделить на тест и трейн набор данных, а также разобраться с указанием тренируемых параметров. 